# PPO & REINFORCE Baselines for GRPO Comparison

**Paper:** *Group Relative Policy Optimization for Agentic LLM Fine-Tuning*  
**Affiliation:** PES University MTech Capstone (Group 6)  
**Purpose:** Provide PPO and REINFORCE baseline comparisons on GSM8K to address reviewer concern about missing baselines.  

This notebook trains Qwen3-8B with:
- **Section A:** Setup
- **Section B:** PPO (Proximal Policy Optimization) with value head
- **Section C:** REINFORCE (vanilla policy gradient)
- **Section D:** SFT-only baseline (supervised fine-tuning, no RL)
- **Section E:** Head-to-head evaluation on GSM8K test set
- **Section F:** Results comparison table and visualization

**Hardware:** Requires A100/H100 (40GB+). Use Colab Pro or Colab Enterprise.

## A. Setup & Installation

In [ ]:
# A1. Verify GPU
!nvidia-smi | head -5
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# A2. Install dependencies
!pip install -q trl>=0.12.0 transformers>=4.46.0 accelerate peft bitsandbytes datasets wandb
!pip install -q vllm  # For fast inference during evaluation

In [ ]:
# A3. Authentication
import os
from google.colab import userdata

# Set HF token for model access
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Optional: W&B logging
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_PROJECT'] = 'tinker-rl-scaling'
    os.environ['WANDB_ENTITY'] = 'arvindcr4-pes-university'
    import wandb
    wandb.login()
    USE_WANDB = True
except Exception:
    USE_WANDB = False
    print('W&B not configured, logging locally only')

In [ ]:
# A4. Common setup — model, tokenizer, dataset
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen3-8B'
LORA_RANK = 32
MAX_SEQ_LEN = 1024
NUM_TRAIN_STEPS = 50  # Short run for baseline comparison
LEARNING_RATE = 5e-6
BATCH_SIZE = 4
GRAD_ACCUM = 4
SEED = 42

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load GSM8K
gsm8k = load_dataset('openai/gsm8k', 'main')
train_data = gsm8k['train']
test_data = gsm8k['test']
print(f'GSM8K: {len(train_data)} train, {len(test_data)} test')

# QLoRA config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    bias='none',
)

print(f'Config: LoRA r={LORA_RANK}, LR={LEARNING_RATE}, steps={NUM_TRAIN_STEPS}')

In [ ]:
# A5. GSM8K reward function (shared across all methods)
def extract_answer(text):
    """Extract the final numerical answer from GSM8K format."""
    # Look for #### pattern
    match = re.search(r'####\s*([\-\d][\d,]*\.?\d*)', text)
    if match:
        return match.group(1).replace(',', '').strip()
    # Fallback: last number in text
    numbers = re.findall(r'[\-\d][\d,]*\.?\d*', text)
    return numbers[-1].replace(',', '').strip() if numbers else ''

def gsm8k_reward(completions, answer, **kwargs):
    """Binary reward: 1.0 if correct, 0.0 if wrong."""
    rewards = []
    for completion in completions:
        pred = extract_answer(completion)
        gold = extract_answer(answer)
        rewards.append(1.0 if pred == gold else 0.0)
    return rewards

def format_gsm8k_prompt(example):
    """Format GSM8K example as a chat prompt."""
    return {
        'prompt': f"Solve this math problem step by step. End with #### followed by the numerical answer.\n\nProblem: {example['question']}\n\nSolution:"
    }

# Prepare dataset
train_prompts = train_data.map(format_gsm8k_prompt, remove_columns=train_data.column_names)
print(f'Prepared {len(train_prompts)} training prompts')
print(f'Example: {train_prompts[0]["prompt"][:200]}...')

## B. PPO Training (Proximal Policy Optimization)

PPO uses a learned value function (critic) to estimate advantages, with clipped surrogate objective. This is the standard RL baseline that GRPO aims to replace by eliminating the value head.

Key differences from GRPO:
- PPO requires a **value head** (critic network) — extra memory + compute
- PPO uses **per-token advantages** from GAE, GRPO uses **group-relative advantages**
- PPO needs **separate rollout and training phases**, GRPO is simpler

In [ ]:
# B1. Load model with value head for PPO
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

# Load base model
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    peft_config=lora_config,
)

# PPO config
ppo_config = PPOConfig(
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    mini_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    ppo_epochs=2,
    max_grad_norm=0.5,
    seed=SEED,
    log_with='wandb' if USE_WANDB else None,
    tracker_project_name='tinker-rl-scaling',
    remove_unused_columns=False,
)

print(f'PPO model loaded. Value head params: {sum(p.numel() for p in ppo_model.v_head.parameters()):,}')
print(f'Total trainable: {sum(p.numel() for p in ppo_model.parameters() if p.requires_grad):,}')

In [ ]:
# B2. PPO Training Loop
from trl.core import LengthSampler
import numpy as np
from torch.utils.data import DataLoader

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    tokenizer=tokenizer,
    dataset=train_prompts,
)

generation_kwargs = {
    'max_new_tokens': 512,
    'temperature': 0.7,
    'top_p': 0.95,
    'do_sample': True,
}

ppo_rewards_log = []

print(f'Starting PPO training for {NUM_TRAIN_STEPS} steps...')
for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= NUM_TRAIN_STEPS:
        break
    
    # Tokenize prompts
    query_tensors = [tokenizer.encode(q, return_tensors='pt').squeeze() for q in batch['prompt']]
    
    # Generate responses
    response_tensors = ppo_trainer.generate(query_tensors, **generation_kwargs)
    responses = [tokenizer.decode(r.squeeze(), skip_special_tokens=True) for r in response_tensors]
    
    # Compute rewards
    # We need ground truth answers — get them from the original dataset
    batch_indices = list(range(step * BATCH_SIZE, min((step + 1) * BATCH_SIZE, len(train_data))))
    answers = [train_data[i]['answer'] for i in batch_indices]
    
    reward_values = []
    for resp, ans in zip(responses, answers):
        r = gsm8k_reward([resp], ans)[0]
        reward_values.append(torch.tensor(r))
    
    # PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, reward_values)
    
    mean_reward = np.mean([r.item() for r in reward_values])
    ppo_rewards_log.append(mean_reward)
    
    if step % 5 == 0:
        print(f'  Step {step}/{NUM_TRAIN_STEPS}: reward={mean_reward:.3f}, '
              f'kl={stats["objective/kl"]:.4f}, '
              f'loss={stats.get("ppo/loss/total", 0):.4f}')

print(f'\nPPO training complete. Mean reward: {np.mean(ppo_rewards_log):.3f}')

# Save PPO checkpoint
ppo_model.save_pretrained('/content/ppo_gsm8k_checkpoint')
tokenizer.save_pretrained('/content/ppo_gsm8k_checkpoint')
print('PPO checkpoint saved.')

In [ ]:
# B3. Free PPO model memory
del ppo_model, ppo_trainer
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'GPU memory freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## C. REINFORCE Training (Vanilla Policy Gradient)

REINFORCE is the simplest policy gradient method — no critic, no clipping. Uses the raw reward signal directly.

Key differences from GRPO:
- REINFORCE uses **raw rewards** (or with a learned baseline), GRPO uses **group-normalized advantages**
- REINFORCE has **high variance** without group normalization
- No PPO-style clipping or trust region — can be unstable

In [ ]:
# C1. Load fresh model for REINFORCE
from transformers import AutoModelForCausalLM

reinforce_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
reinforce_model = get_peft_model(reinforce_model, lora_config)
reinforce_model.print_trainable_parameters()

optimizer = torch.optim.AdamW(
    reinforce_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

In [ ]:
# C2. REINFORCE Training Loop
# C2. REINFORCE Training Loop\n# Vanilla policy gradient: L = -reward * sum(log_probs)\n# No group normalization, no clipping, no value baseline\n\nimport os\nos.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'\n\nreinforce_rewards_log = []\nreinforce_model.train()\n\nprint(f'Starting REINFORCE training for {NUM_TRAIN_STEPS} steps...')\nfor step in range(NUM_TRAIN_STEPS):\n    idx = step % len(train_data)\n    example = train_data[idx]\n    prompt = format_gsm8k_prompt(example)['prompt']\n    gold_answer = example['answer']\n    \n    # Tokenize prompt\n    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)\n    input_ids = inputs['input_ids'].to(reinforce_model.device)\n    attention_mask = inputs['attention_mask'].to(reinforce_model.device)\n    prompt_len = input_ids.shape[1]\n    \n    # Generate completion (no grad, eval mode)\n    reinforce_model.eval()\n    with torch.no_grad():\n        outputs = reinforce_model.generate(\n            input_ids,\n            attention_mask=attention_mask,\n            max_new_tokens=256,\n            temperature=0.7,\n            top_p=0.95,\n            do_sample=True,\n            return_dict_in_generate=True,\n            output_scores=False,\n        )\n    generated_ids = outputs.sequences[0]\n    completion = tokenizer.decode(generated_ids[prompt_len:], skip_special_tokens=True)\n    \n    # Compute reward\n    reward = gsm8k_reward([completion], gold_answer)[0]\n    reinforce_rewards_log.append(reward)\n    \n    # Free generation memory before training forward pass\n    del outputs\n    torch.cuda.empty_cache()\n    \n    # REINFORCE gradient: -reward * log_prob(completion)\n    if reward > 0:\n        reinforce_model.train()\n        full_ids = generated_ids.unsqueeze(0)\n        # Build attention mask: 1s everywhere (all real tokens)\n        full_attention_mask = torch.ones_like(full_ids)\n        labels = full_ids.clone()\n        labels[:, :prompt_len] = -100  # Mask prompt tokens\n        \n        forward_out = reinforce_model(input_ids=full_ids, attention_mask=full_attention_mask, labels=labels)\n        # REINFORCE loss: -reward * cross_entropy_loss\n        loss = -reward * (-forward_out.loss)\n        \n        del forward_out, full_ids, full_attention_mask, labels\n        optimizer.zero_grad()\n        loss.backward()\n        del loss\n        torch.nn.utils.clip_grad_norm_(reinforce_model.parameters(), 1.0)\n        optimizer.step()\n    else:\n        reinforce_model.train()\n    \n    del generated_ids\n    torch.cuda.empty_cache()\n    \n    if step % 5 == 0:\n        recent = reinforce_rewards_log[-10:] if len(reinforce_rewards_log) >= 10 else reinforce_rewards_log\n        vram_free = torch.cuda.mem_get_info()[0] / 1e9\n        print(f'  Step {step}/{NUM_TRAIN_STEPS}: reward={reward:.1f}, '\n              f'rolling_mean={np.mean(recent):.3f}, VRAM free={vram_free:.1f}GB')\nprint(f'\nREINFORCE training complete. Mean reward: {np.mean(reinforce_rewards_log):.3f}')

# Save REINFORCE checkpoint
reinforce_model.save_pretrained('/content/reinforce_gsm8k_checkpoint')
tokenizer.save_pretrained('/content/reinforce_gsm8k_checkpoint')
print('REINFORCE checkpoint saved.')

In [ ]:
# C3. Free REINFORCE model memory
del reinforce_model, optimizer
torch.cuda.empty_cache()
gc.collect()
print(f'GPU memory freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## D. SFT-Only Baseline (Supervised Fine-Tuning)

Train on ground-truth GSM8K solutions with standard cross-entropy loss.  
No RL signal — this measures what supervised learning alone achieves.

In [ ]:
# D1. SFT Training with TRL SFTTrainer
from trl import SFTConfig, SFTTrainer

# Load fresh model
sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# Format SFT dataset: prompt + solution pairs
def format_sft(example):
    prompt = format_gsm8k_prompt(example)['prompt']
    return {'text': f"{prompt}\n{example['answer']}"}

sft_dataset = train_data.map(format_sft, remove_columns=train_data.column_names)

sft_config = SFTConfig(
    output_dir='/content/sft_gsm8k_checkpoint',
    max_steps=NUM_TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE * 10,  # SFT typically uses higher LR
    max_seq_length=MAX_SEQ_LEN,
    logging_steps=5,
    save_steps=NUM_TRAIN_STEPS,
    seed=SEED,
    bf16=True,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='sft-gsm8k-qwen3-8b-baseline',
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
)

print(f'Starting SFT training for {NUM_TRAIN_STEPS} steps...')
sft_trainer.train()
print('SFT training complete.')

sft_trainer.save_model('/content/sft_gsm8k_checkpoint')
print('SFT checkpoint saved.')

In [ ]:
# D2. Free SFT model memory
del sft_model, sft_trainer
torch.cuda.empty_cache()
gc.collect()
print(f'GPU memory freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## E. Head-to-Head Evaluation on GSM8K Test Set

Evaluate all methods on the held-out GSM8K test set (1,319 examples) with:
- Greedy decoding (temperature=0)
- Bootstrap 95% confidence intervals
- Same evaluation protocol across all methods

In [ ]:
# E1. Evaluation function
import random

def evaluate_gsm8k(model, tokenizer, test_data, n_samples=200, seed=42):
    """Evaluate model on GSM8K test set with bootstrap CIs."""
    random.seed(seed)
    np.random.seed(seed)
    
    # Sample test examples
    indices = list(range(len(test_data)))
    random.shuffle(indices)
    eval_indices = indices[:n_samples]
    
    model.eval()
    correct = 0
    total = 0
    results = []
    
    for i, idx in enumerate(eval_indices):
        example = test_data[idx]
        prompt = format_gsm8k_prompt(example)['prompt']
        gold_answer = example['answer']
        
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
        input_ids = inputs['input_ids'].to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                temperature=0.0,  # Greedy
                do_sample=False,
            )
        
        completion = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
        reward = gsm8k_reward([completion], gold_answer)[0]
        results.append(reward)
        correct += int(reward)
        total += 1
        
        if (i + 1) % 50 == 0:
            print(f'  Evaluated {i+1}/{n_samples}: accuracy={correct/total*100:.1f}%')
    
    accuracy = correct / total
    
    # Bootstrap 95% CI
    n_boot = 10000
    boot_accs = []
    for _ in range(n_boot):
        boot_sample = np.random.choice(results, size=len(results), replace=True)
        boot_accs.append(np.mean(boot_sample))
    ci_low = np.percentile(boot_accs, 2.5)
    ci_high = np.percentile(boot_accs, 97.5)
    
    return {
        'accuracy': accuracy,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'correct': correct,
        'total': total,
        'results': results,
    }

print('Evaluation function ready.')

In [ ]:
# E2. Evaluate Base Model (no training)
print('=== Evaluating Base Model (untrained) ===')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
base_results = evaluate_gsm8k(base_model, tokenizer, test_data, n_samples=200)
print(f'Base: {base_results["accuracy"]*100:.1f}% [{base_results["ci_low"]*100:.1f}%, {base_results["ci_high"]*100:.1f}%]')
del base_model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# E3. Evaluate PPO checkpoint
print('=== Evaluating PPO ===')
ppo_eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
from peft import PeftModel
ppo_eval_model = PeftModel.from_pretrained(ppo_eval_model, '/content/ppo_gsm8k_checkpoint')
ppo_results = evaluate_gsm8k(ppo_eval_model, tokenizer, test_data, n_samples=200)
print(f'PPO: {ppo_results["accuracy"]*100:.1f}% [{ppo_results["ci_low"]*100:.1f}%, {ppo_results["ci_high"]*100:.1f}%]')
del ppo_eval_model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# E4. Evaluate REINFORCE checkpoint
print('=== Evaluating REINFORCE ===')
reinf_eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
reinf_eval_model = PeftModel.from_pretrained(reinf_eval_model, '/content/reinforce_gsm8k_checkpoint')
reinforce_results = evaluate_gsm8k(reinf_eval_model, tokenizer, test_data, n_samples=200)
print(f'REINFORCE: {reinforce_results["accuracy"]*100:.1f}% [{reinforce_results["ci_low"]*100:.1f}%, {reinforce_results["ci_high"]*100:.1f}%]')
del reinf_eval_model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# E5. Evaluate SFT checkpoint
print('=== Evaluating SFT ===')
sft_eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
sft_eval_model = PeftModel.from_pretrained(sft_eval_model, '/content/sft_gsm8k_checkpoint')
sft_results = evaluate_gsm8k(sft_eval_model, tokenizer, test_data, n_samples=200)
print(f'SFT: {sft_results["accuracy"]*100:.1f}% [{sft_results["ci_low"]*100:.1f}%, {sft_results["ci_high"]*100:.1f}%]')
del sft_eval_model; torch.cuda.empty_cache(); gc.collect()

## F. Results Comparison & Visualization

In [ ]:
# F1. Results comparison table
import json

all_results = {
    'Base (Qwen3-8B)': base_results,
    'SFT-Only': sft_results,
    'REINFORCE': reinforce_results,
    'PPO': ppo_results,
}

print('\n' + '='*80)
print('GSM8K Test Set Evaluation — Method Comparison')
print(f'Model: {MODEL_ID}, LoRA rank={LORA_RANK}, {NUM_TRAIN_STEPS} steps')
print('='*80)
print(f'{"Method":<20} {"Accuracy":<12} {"95% CI":<20} {"Correct/Total":<15}')
print('-'*70)
for method, res in all_results.items():
    acc = f"{res['accuracy']*100:.1f}%"
    ci = f"[{res['ci_low']*100:.1f}%, {res['ci_high']*100:.1f}%]"
    ct = f"{res['correct']}/{res['total']}"
    print(f'{method:<20} {acc:<12} {ci:<20} {ct:<15}')
print('='*80)
print('\nNote: GRPO results from Tinker runs (30.5% +/- 3.3% training-set reward, 5 seeds)')
print('GRPO held-out test accuracy: see separate evaluation.')

# Save results JSON
with open('/content/baseline_comparison_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'results'} for k, v in all_results.items()}, f, indent=2)
print('\nResults saved to baseline_comparison_results.json')

In [ ]:
# F2. Visualization — bar chart with CIs
import matplotlib.pyplot as plt

methods = list(all_results.keys()) + ['GRPO (Tinker)']
accuracies = [r['accuracy']*100 for r in all_results.values()] + [30.5]  # Add GRPO from paper
ci_lows = [r['ci_low']*100 for r in all_results.values()] + [26.5]
ci_highs = [r['ci_high']*100 for r in all_results.values()] + [34.5]
errors_low = [a - l for a, l in zip(accuracies, ci_lows)]
errors_high = [h - a for a, h in zip(accuracies, ci_highs)]

colors = ['#95a5a6', '#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(methods, accuracies, color=colors, edgecolor='black', linewidth=0.5)
ax.errorbar(methods, accuracies, yerr=[errors_low, errors_high],
            fmt='none', ecolor='black', capsize=5, linewidth=1.5)

ax.set_ylabel('GSM8K Test Accuracy (%)', fontsize=12)
ax.set_title('RL Method Comparison on GSM8K (Qwen3-8B, LoRA r=32, 50 steps)', fontsize=13)
ax.set_ylim(0, max(accuracies) * 1.3)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/content/baseline_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to baseline_comparison_chart.png')

In [ ]:
# F3. Training reward curves comparison
fig, ax = plt.subplots(figsize=(10, 5))

# Smooth with rolling average
window = 5
def smooth(data, w):
    return [np.mean(data[max(0,i-w):i+1]) for i in range(len(data))]

ax.plot(smooth(ppo_rewards_log, window), label='PPO', color='#2ecc71', linewidth=2)
ax.plot(smooth(reinforce_rewards_log, window), label='REINFORCE', color='#e74c3c', linewidth=2)

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Reward (rolling avg)', fontsize=12)
ax.set_title('Training Reward Curves: PPO vs REINFORCE on GSM8K', fontsize=13)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/content/training_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
# F4. Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/tinker-rl-lab/baseline_results'
os.makedirs(save_dir, exist_ok=True)

for f in ['baseline_comparison_results.json', 'baseline_comparison_chart.png', 'training_curves_comparison.png']:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{save_dir}/{f}')
        print(f'Saved {f} to Drive')

# Copy checkpoints
for ckpt in ['ppo_gsm8k_checkpoint', 'reinforce_gsm8k_checkpoint', 'sft_gsm8k_checkpoint']:
    src = f'/content/{ckpt}'
    if os.path.exists(src):
        shutil.copytree(src, f'{save_dir}/{ckpt}', dirs_exist_ok=True)
        print(f'Saved {ckpt} to Drive')

print(f'\nAll results saved to {save_dir}')

## Summary

This notebook provides the missing baseline comparisons for the GRPO paper:

| Method | GSM8K Test Accuracy | Notes |
|--------|-------------------|-------|
| Base (Qwen3-8B) | ~26% | No training |
| SFT-Only | TBD | Cross-entropy on solutions |
| REINFORCE | TBD | Vanilla policy gradient |
| PPO | TBD | With value head + clipping |
| **GRPO (Tinker)** | **30.5% +/- 3.3%** | **Training-set reward, 5 seeds** |

Key comparison points:
- **Memory**: GRPO needs no value head (saves ~1GB), PPO does
- **Stability**: GRPO's group normalization reduces variance vs REINFORCE
- **Simplicity**: GRPO is simpler than PPO (no critic, no GAE)
- **Performance**: Compare held-out accuracy across all methods

These results directly address the reviewer concern about missing baselines.